In [1]:
import os
import pickle
import kagglehub
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from PIL import Image
from sklearn.model_selection import train_test_split


# ============================================================
# Configuration
# ============================================================

BATCH_SIZE = 128
NUM_WORKERS = 4
SEED = 6304


# ============================================================
# Download datasets
# ============================================================

cifar100_path = kagglehub.dataset_download(
    "fedesoriano/cifar100"
)

cifar10_path = kagglehub.dataset_download(
    "ayush1220/cifar10"
)

print("CIFAR-100 path:", cifar100_path)
print("CIFAR-10 path:", cifar10_path)


# ============================================================
# Transform
# ============================================================

transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# ============================================================
# Helper functions
# ============================================================

def unpickle(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="bytes")


def find_file(root, filename):
    for directory, _, files in os.walk(root):
        if filename in files:
            return os.path.join(directory, filename)

    raise FileNotFoundError(
        f"{filename} not found in {root}"
    )


# ============================================================
# Custom CIFAR-100 Dataset
# ============================================================

class CIFAR100Dataset(Dataset):

    def __init__(self, data, targets, transform=None):
        self.data = data
        self.targets = np.asarray(targets)
        self.transform = transform

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):

        image = Image.fromarray(
            self.data[index]
        )

        target = int(
            self.targets[index]
        )

        if self.transform:
            image = self.transform(image)

        return image, target


# ============================================================
# Load CIFAR-100 test set
# ============================================================

cifar100_test_file = find_file(
    cifar100_path,
    "test"
)

cifar100_meta_file = find_file(
    cifar100_path,
    "meta"
)

cifar100_test = unpickle(
    cifar100_test_file
)

cifar100_meta = unpickle(
    cifar100_meta_file
)


# ============================================================
# CIFAR-100 images and labels
# ============================================================

cifar100_data = cifar100_test[b"data"]

cifar100_targets = np.asarray(
    cifar100_test[b"fine_labels"]
)

cifar100_data = cifar100_data.reshape(
    -1, 3, 32, 32
).transpose(
    0, 2, 3, 1
)


# ============================================================
# CIFAR-100 class names and mappings
# ============================================================

cifar100_classes = [
    name.decode("utf-8")
    for name in cifar100_meta[b"fine_label_names"]
]


# Class name -> original CIFAR-100 label
cifar100_class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(
        cifar100_classes
    )
}


# Original CIFAR-100 label -> class name
cifar100_idx_to_class = {
    idx: class_name
    for idx, class_name in enumerate(
        cifar100_classes
    )
}


# ============================================================
# Construct CIFAR-100 dataset
# ============================================================

cifar100_dataset = CIFAR100Dataset(
    cifar100_data,
    cifar100_targets,
    transform=transform,
)


# ============================================================
# Near unknown classes
# ============================================================

near_unknown_classes = [
    "bus",
    "pickup_truck",
    "motorcycle",
    "tractor",
    "wolf",
    "fox",
    "leopard",
    "camel",
]


# ============================================================
# Far unknown classes
# ============================================================

far_unknown_classes = [
    "bottle",
    "bowl",
    "chair",
    "clock",
    "keyboard",
    "mushroom",
    "sunflower",
    "wardrobe",
]




near_label_to_class = {
    cifar100_class_to_idx[class_name]: class_name
    for class_name in near_unknown_classes
}


far_label_to_class = {
    cifar100_class_to_idx[class_name]: class_name
    for class_name in far_unknown_classes
}


# ============================================================
# Create subset using class names
# ============================================================

def subset_by_classes(
    dataset,
    class_names,
    class_to_idx
):

    class_ids = {
        class_to_idx[name]
        for name in class_names
    }

    indices = [
        i
        for i, target in enumerate(dataset.targets)
        if target in class_ids
    ]

    return Subset(
        dataset,
        indices
    )


# ============================================================
# Near / Far unknown datasets
# ============================================================

NearUnknownDataset = subset_by_classes(
    cifar100_dataset,
    near_unknown_classes,
    cifar100_class_to_idx,
)


FarUnknownDataset = subset_by_classes(
    cifar100_dataset,
    far_unknown_classes,
    cifar100_class_to_idx,
)


# ============================================================
# Near / Far unknown DataLoaders
# ============================================================

NearUnknownDataLoader = DataLoader(
    NearUnknownDataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


FarUnknownDataLoader = DataLoader(
    FarUnknownDataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ============================================================
# CIFAR-10
# ============================================================

cifar10_train_path = os.path.join(
    cifar10_path,
    "cifar10",
    "train",
)


cifar10_dataset = datasets.ImageFolder(
    root=cifar10_train_path,
    transform=transform,
)




cifar10_class_to_idx = dict(
    cifar10_dataset.class_to_idx
)


cifar10_idx_to_class = {
    idx: class_name
    for class_name, idx
    in cifar10_class_to_idx.items()
}


# Model output index -> CIFAR-10 class name
class_labels = [
    cifar10_idx_to_class[idx]
    for idx in range(
        len(cifar10_idx_to_class)
    )
]


indices = np.arange(
    len(cifar10_dataset)
)

targets = np.asarray(
    cifar10_dataset.targets
)


# ============================================================
# First split
#
# Entire CIFAR-10:
# 90% -> remaining training data
# 10% -> test
# ============================================================

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.10,
    stratify=targets,
    random_state=SEED,
)


# ============================================================
# Second split
#
# Remaining training data:
# 90% -> train
# 10% -> validation
# ============================================================

train_indices, val_indices = train_test_split(
    train_indices,
    test_size=0.10,
    stratify=targets[train_indices],
    random_state=SEED,
)


# ============================================================
# Create CIFAR-10 datasets
# ============================================================

train_dataset = Subset(
    cifar10_dataset,
    train_indices,
)


val_dataset = Subset(
    cifar10_dataset,
    val_indices,
)


test_dataset = Subset(
    cifar10_dataset,
    test_indices,
)


# ============================================================
# CIFAR-10 DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


# ============================================================
# Dataset sizes
# ============================================================

print("\nDataset sizes")
print("-" * 50)

print(
    f"CIFAR-10 Train:       "
    f"{len(train_dataset)}"
)

print(
    f"CIFAR-10 Validation:  "
    f"{len(val_dataset)}"
)

print(
    f"CIFAR-10 Test:        "
    f"{len(test_dataset)}"
)

print(
    f"Near Unknown:         "
    f"{len(NearUnknownDataset)}"
)

print(
    f"Far Unknown:          "
    f"{len(FarUnknownDataset)}"
)


# ============================================================
# Display mappings
# ============================================================

print("\nCIFAR-10 label mapping")
print("-" * 50)

for label, class_name in sorted(
    cifar10_idx_to_class.items()
):
    print(
        f"{label:2d} -> {class_name}"
    )


print("\nNear-unknown CIFAR-100 label mapping")
print("-" * 50)

for label, class_name in sorted(
    near_label_to_class.items()
):
    print(
        f"{label:2d} -> {class_name}"
    )


print("\nFar-unknown CIFAR-100 label mapping")
print("-" * 50)

for label, class_name in sorted(
    far_label_to_class.items()
):
    print(
        f"{label:2d} -> {class_name}"
    )

CIFAR-100 path: /root/.cache/kagglehub/datasets/fedesoriano/cifar100/versions/1
CIFAR-10 path: /root/.cache/kagglehub/datasets/ayush1220/cifar10/versions/2

Dataset sizes
--------------------------------------------------
CIFAR-10 Train:       40500
CIFAR-10 Validation:  4500
CIFAR-10 Test:        5000
Near Unknown:         800
Far Unknown:          800

CIFAR-10 label mapping
--------------------------------------------------
 0 -> airplane
 1 -> automobile
 2 -> bird
 3 -> cat
 4 -> deer
 5 -> dog
 6 -> frog
 7 -> horse
 8 -> ship
 9 -> truck

Near-unknown CIFAR-100 label mapping
--------------------------------------------------
13 -> bus
15 -> camel
34 -> fox
42 -> leopard
48 -> motorcycle
58 -> pickup_truck
89 -> tractor
97 -> wolf

Far-unknown CIFAR-100 label mapping
--------------------------------------------------
 9 -> bottle
10 -> bowl
20 -> chair
22 -> clock
39 -> keyboard
51 -> mushroom
82 -> sunflower
94 -> wardrobe


# Vanilla Closed-Set Baseline

In [2]:
import torch
import torch.nn as nn
import torchvision.models as models

from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR


# ============================================================
# Number of known CIFAR-10 classes
# ============================================================

NUM_KNOWN_CLASSES = len(
    cifar10_idx_to_class
)


# ============================================================
# Verify model output mapping
# ============================================================

print("Closed-set model output mapping")
print("-" * 50)

for idx in range(NUM_KNOWN_CLASSES):

    print(
        f"Output {idx:2d} -> "
        f"{cifar10_idx_to_class[idx]}"
    )


# ============================================================
# ResNet18
# ============================================================

resnet18_baseline = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)


# Replace ImageNet classifier with CIFAR-10 classifier
resnet18_baseline.fc = nn.Linear(
    resnet18_baseline.fc.in_features,
    NUM_KNOWN_CLASSES
)


# ============================================================
# Loss
# ============================================================

loss_fn = nn.CrossEntropyLoss()


# ============================================================
# Optimizer
# ============================================================

optim = SGD(
    resnet18_baseline.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
)


# ============================================================
# Learning-rate scheduler
# ============================================================

scheduler = CosineAnnealingLR(
    optim,
    T_max=100,
    eta_min=1e-6,
)

Closed-set model output mapping
--------------------------------------------------
Output  0 -> airplane
Output  1 -> automobile
Output  2 -> bird
Output  3 -> cat
Output  4 -> deer
Output  5 -> dog
Output  6 -> frog
Output  7 -> horse
Output  8 -> ship
Output  9 -> truck


In [3]:
import torch


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


resnet18_baseline = (
    resnet18_baseline.to(device)
)


# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(
    epoch_index,
    model,
    loader,
    optimizer,
    loss_fn
):

    model.train()

    running_loss = 0.0


    for images, labels in loader:

        images = images.to(device)

        labels = labels.to(device)


        # --------------------------------------------
        # Reset gradients
        # --------------------------------------------

        optimizer.zero_grad()


        # --------------------------------------------
        # Forward
        # --------------------------------------------

        logits = model(images)


        # --------------------------------------------
        # Loss
        # --------------------------------------------

        loss = loss_fn(
            logits,
            labels
        )


        # --------------------------------------------
        # Backward
        # --------------------------------------------

        loss.backward()


        # --------------------------------------------
        # Update parameters
        # --------------------------------------------

        optimizer.step()


        running_loss += (
            loss.item()
        )


    avg_loss = (
        running_loss / len(loader)
    )


    return model, avg_loss


# ============================================================
# Evaluation
# ============================================================

def eval_model(
    model,
    loader,
    loss_fn
):

    model.eval()


    running_val_loss = 0.0

    correct = 0

    total = 0


    with torch.inference_mode():

        for images, targets in loader:

            images = images.to(device)

            targets = targets.to(device)


            # ----------------------------------------
            # Forward
            # ----------------------------------------

            logits = model(images)


            # ----------------------------------------
            # Loss
            # ----------------------------------------

            loss = loss_fn(
                logits,
                targets
            )

            running_val_loss += (
                loss.item()
            )


            # ----------------------------------------
            # Prediction
            # ----------------------------------------

            predicted = logits.argmax(
                dim=1
            )


            total += targets.size(0)


            correct += (
                predicted == targets
            ).sum().item()


    avg_val_loss = (
        running_val_loss /
        len(loader)
    )


    accuracy = (
        100.0 * correct / total
    )


    return (
        avg_val_loss,
        accuracy
    )

Device: cuda


In [4]:
# ============================================================
# Training configuration
# ============================================================

num_epochs = 100


# ============================================================
# Training loop
# ============================================================

for epoch in range(num_epochs):


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    resnet18_baseline, train_loss = train_one_epoch(
        epoch,
        resnet18_baseline,
        train_loader,
        optim,
        loss_fn
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_loss, val_accuracy = eval_model(
        resnet18_baseline,
        val_loader,
        loss_fn
    )


    # --------------------------------------------------------
    # Learning-rate update
    # --------------------------------------------------------

    scheduler.step()


    current_lr = (
        optim.param_groups[0]["lr"]
    )


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}% | "
        f"LR: {current_lr:.6f}"
    )


# ============================================================
# Save trained CLOSED-SET classifier
# ============================================================

torch.save(
    resnet18_baseline.state_dict(),
    "resnet18_baseline.pth"
)


print(
    "Baseline model saved successfully."
)

Epoch [1/100] | Train Loss: 3.0420 | Val Loss: 1.9311 | Val Accuracy: 25.24% | LR: 0.099975
Epoch [2/100] | Train Loss: 1.8222 | Val Loss: 1.7696 | Val Accuracy: 34.76% | LR: 0.099901
Epoch [3/100] | Train Loss: 1.6189 | Val Loss: 1.5791 | Val Accuracy: 43.11% | LR: 0.099778
Epoch [4/100] | Train Loss: 1.4435 | Val Loss: 1.4736 | Val Accuracy: 47.09% | LR: 0.099606
Epoch [5/100] | Train Loss: 1.3325 | Val Loss: 1.3521 | Val Accuracy: 51.56% | LR: 0.099384
Epoch [6/100] | Train Loss: 1.2572 | Val Loss: 1.2718 | Val Accuracy: 55.96% | LR: 0.099114
Epoch [7/100] | Train Loss: 1.1698 | Val Loss: 1.2506 | Val Accuracy: 55.73% | LR: 0.098796
Epoch [8/100] | Train Loss: 1.1188 | Val Loss: 1.1947 | Val Accuracy: 57.84% | LR: 0.098429
Epoch [9/100] | Train Loss: 1.0800 | Val Loss: 1.1979 | Val Accuracy: 59.04% | LR: 0.098015
Epoch [10/100] | Train Loss: 1.0380 | Val Loss: 1.1007 | Val Accuracy: 61.09% | LR: 0.097553
Epoch [11/100] | Train Loss: 1.0130 | Val Loss: 1.0986 | Val Accuracy: 62.16% |

In [5]:
# ============================================================
# CIFAR-10 mapping
# ============================================================

print("=" * 65)
print("CIFAR-10 KNOWN CLASSES")
print("=" * 65)

print(
    "These labels correspond to the output indices "
    "of the closed-set classifier.\n"
)


for label, class_name in sorted(
    cifar10_idx_to_class.items()
):

    print(
        f"Label {label:2d} -> "
        f"{class_name}"
    )


# ============================================================
# Near-unknown mapping
# ============================================================

print("\n" + "=" * 65)
print("CIFAR-100 NEAR-UNKNOWN CLASSES")
print("=" * 65)

print(
    "These are ORIGINAL CIFAR-100 fine-label IDs.\n"
)


for label, class_name in sorted(
    near_label_to_class.items()
):

    print(
        f"Label {label:2d} -> "
        f"{class_name}"
    )


# ============================================================
# Far-unknown mapping
# ============================================================

print("\n" + "=" * 65)
print("CIFAR-100 FAR-UNKNOWN CLASSES")
print("=" * 65)

print(
    "These are ORIGINAL CIFAR-100 fine-label IDs.\n"
)


for label, class_name in sorted(
    far_label_to_class.items()
):

    print(
        f"Label {label:2d} -> "
        f"{class_name}"
    )




CIFAR-10 KNOWN CLASSES
These labels correspond to the output indices of the closed-set classifier.

Label  0 -> airplane
Label  1 -> automobile
Label  2 -> bird
Label  3 -> cat
Label  4 -> deer
Label  5 -> dog
Label  6 -> frog
Label  7 -> horse
Label  8 -> ship
Label  9 -> truck

CIFAR-100 NEAR-UNKNOWN CLASSES
These are ORIGINAL CIFAR-100 fine-label IDs.

Label 13 -> bus
Label 15 -> camel
Label 34 -> fox
Label 42 -> leopard
Label 48 -> motorcycle
Label 58 -> pickup_truck
Label 89 -> tractor
Label 97 -> wolf

CIFAR-100 FAR-UNKNOWN CLASSES
These are ORIGINAL CIFAR-100 fine-label IDs.

Label  9 -> bottle
Label 10 -> bowl
Label 20 -> chair
Label 22 -> clock
Label 39 -> keyboard
Label 51 -> mushroom
Label 82 -> sunflower
Label 94 -> wardrobe


In [6]:
import copy
import torch
import torch.nn as nn


# ============================================================
# IMPORTANT
#
# Keep resnet18_baseline as the trained 10-class classifier.
#
# Make a separate copy for feature extraction.
# ============================================================

feature_extractor = copy.deepcopy(
    resnet18_baseline
)


# Remove ONLY the copied model's classifier.
#
# ResNet18 now returns its 512-dimensional
# penultimate representation.

feature_extractor.fc = nn.Identity()


feature_extractor = (
    feature_extractor.to(device)
)

feature_extractor.eval()


# ============================================================
# Feature extraction
# ============================================================

def extract_features(
    model,
    loader,
    device
):
    """
    Extract 512-dimensional ResNet18 features.

    Labels are NOT changed.

    CIFAR-10:
        Labels are CIFAR-10 IDs (0-9).

    Near/Far Unknown:
        Labels remain their original CIFAR-100
        fine-label IDs.
    """

    features_list = []

    labels_list = []


    with torch.inference_mode():

        for images, labels in loader:

            images = images.to(device)


            # ----------------------------------------
            # Extract [batch_size, 512] features
            # ----------------------------------------

            features = model(images)


            # ----------------------------------------
            # Move results to CPU
            # ----------------------------------------

            features_list.append(
                features.cpu()
            )

            labels_list.append(
                labels.cpu()
            )


    # --------------------------------------------------------
    # Combine batches
    # --------------------------------------------------------

    features = torch.cat(
        features_list,
        dim=0
    )


    labels = torch.cat(
        labels_list,
        dim=0
    )


    return (
        features,
        labels
    )


# ============================================================
# CIFAR-10 Train features
# ============================================================

train_features, train_labels = extract_features(
    feature_extractor,
    train_loader,
    device
)


# ============================================================
# CIFAR-10 Validation features
# ============================================================

val_features, val_labels = extract_features(
    feature_extractor,
    val_loader,
    device
)


# ============================================================
# CIFAR-10 Test features
# ============================================================

test_features, test_labels = extract_features(
    feature_extractor,
    test_loader,
    device
)


# ============================================================
# Near Unknown features
# ============================================================

near_unknown_features, near_unknown_labels = (
    extract_features(
        feature_extractor,
        NearUnknownDataLoader,
        device
    )
)


# ============================================================
# Far Unknown features
# ============================================================

far_unknown_features, far_unknown_labels = (
    extract_features(
        feature_extractor,
        FarUnknownDataLoader,
        device
    )
)



In [7]:
import torch
import torch.nn.functional as F
import pandas as pd

def analyze_unknown_classes(
    classifier,
    loader,
    unknown_label_to_class,
    known_label_to_class,
    device,
    dataset_name
):
    classifier.eval()
    seen_classes = set()
    results = []

    with torch.inference_mode():
        for images, labels in loader:
            for i in range(len(labels)):
                true_label = int(labels[i].item())

                if true_label in seen_classes:
                    continue

                seen_classes.add(true_label)
                actual_class = unknown_label_to_class[true_label]

                image = images[i].unsqueeze(0).to(device)
                logits = classifier(image)

                expected_outputs = len(known_label_to_class)

                assert logits.ndim == 2
                assert logits.shape[1] == expected_outputs

                probabilities = F.softmax(logits, dim=1)
                confidence, predicted_label = probabilities.max(dim=1)

                predicted_label = int(predicted_label.item())
                confidence = float(confidence.item())
                predicted_class = known_label_to_class[predicted_label]

                results.append({
                    "Dataset": dataset_name,
                    "True Label ID": true_label,
                    "Actual Unknown Class": actual_class,
                    "Predicted Label ID": predicted_label,
                    "Predicted Known Class": predicted_class,
                    "Confidence (%)": round(confidence * 100, 2),
                })

                if len(seen_classes) == len(unknown_label_to_class):
                    return results

    return results


near_results = analyze_unknown_classes(
    resnet18_baseline,
    NearUnknownDataLoader,
    near_label_to_class,
    cifar10_idx_to_class,
    device,
    "NearUnknown"
)

far_results = analyze_unknown_classes(
    resnet18_baseline,
    FarUnknownDataLoader,
    far_label_to_class,
    cifar10_idx_to_class,
    device,
    "FarUnknown"
)

failure_analysis = pd.DataFrame(
    near_results + far_results
)[[
    "Dataset",
    "True Label ID",
    "Actual Unknown Class",
    "Predicted Label ID",
    "Predicted Known Class",
    "Confidence (%)",
]]

display(failure_analysis)

,Dataset,True Label ID,Actual Unknown Class,Predicted Label ID,Predicted Known Class,Confidence (%)
0,NearUnknown,15,camel,2,bird,95.53
1,NearUnknown,97,wolf,5,dog,99.44
2,NearUnknown,58,pickup_truck,9,truck,54.65
3,NearUnknown,42,leopard,6,frog,88.15
4,NearUnknown,89,tractor,8,ship,94.24
5,NearUnknown,13,bus,9,truck,99.97
6,NearUnknown,34,fox,3,cat,95.57
7,NearUnknown,48,motorcycle,2,bird,69.74
8,FarUnknown,51,mushroom,1,automobile,75.02
9,FarUnknown,39,keyboard,8,ship,95.55


In [8]:
# ============================================================
# CIFAR-10 train loader WITHOUT augmentation
# Same exact training images / indices as train_loader
# ============================================================

no_aug_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# Same underlying CIFAR-10 images, but without
# RandomCrop and RandomHorizontalFlip
cifar10_dataset_no_aug = datasets.ImageFolder(
    root=cifar10_train_path,
    transform=no_aug_transform,
)


# IMPORTANT:
# Reuse the exact same train_indices generated earlier.
train_dataset_no_aug = Subset(
    cifar10_dataset_no_aug,
    train_indices,
)


train_loader_no_aug = DataLoader(
    train_dataset_no_aug,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

In [9]:
train_features_no_aug, train_labels_no_aug = extract_features(
    feature_extractor,
    train_loader_no_aug,
    device
)

train_features_no_aug = train_features_no_aug.float()
train_labels_no_aug = train_labels_no_aug.long()

num_classes = 10
feature_dim = train_features_no_aug.shape[1]

class_means = torch.stack([
    train_features_no_aug[
        train_labels_no_aug == c
    ].mean(dim=0)
    for c in range(num_classes)
])

print("Class means shape:", class_means.shape)

centered_features = (
    train_features_no_aug
    - class_means[train_labels_no_aug]
)

N = centered_features.shape[0]

Sigma = (
    centered_features.T @ centered_features
) / (N - num_classes)

Sigma = Sigma + (
    1e-6 * torch.eye(
        feature_dim,
        dtype=Sigma.dtype
    )
)

print("Sigma shape:", Sigma.shape)

Sigma_inv = torch.linalg.inv(Sigma)

print("Sigma inverse shape:", Sigma_inv.shape)

Class means shape: torch.Size([10, 512])
Sigma shape: torch.Size([512, 512])
Sigma inverse shape: torch.Size([512, 512])


In [10]:
def feature_to_logits(feature):
    feature = feature.float().to(device)

    with torch.inference_mode():
        logits = resnet18_baseline.fc(feature)

    return logits


def msp_unknown_score(feature):
    logits = feature_to_logits(feature)

    probabilities = torch.softmax(
        logits,
        dim=-1
    )

    score = 1.0 - probabilities.max()

    return score.item()


def mls_unknown_score(feature):
    logits = feature_to_logits(feature)

    score = -logits.max()

    return score.item()


def energy_unknown_score(feature):
    logits = feature_to_logits(feature)

    score = -torch.logsumexp(
        logits,
        dim=-1
    )

    return score.item()


def mahalanobis_unknown_score(feature):
    feature = feature.float().cpu()

    differences = (
        feature.unsqueeze(0)
        - class_means
    )

    distances = torch.einsum(
        "cd,de,ce->c",
        differences,
        Sigma_inv,
        differences
    )

    score = distances.min()

    return score.item()

In [11]:
import torch

msp_val_scores = torch.tensor([
    msp_unknown_score(feature)
    for feature in val_features
])

mls_val_scores = torch.tensor([
    mls_unknown_score(feature)
    for feature in val_features
])

energy_val_scores = torch.tensor([
    energy_unknown_score(feature)
    for feature in val_features
])

mahalanobis_val_scores = torch.tensor([
    mahalanobis_unknown_score(feature)
    for feature in val_features
])

tau_msp = torch.quantile(msp_val_scores, 0.95).item()
tau_mls = torch.quantile(mls_val_scores, 0.95).item()
tau_energy = torch.quantile(energy_val_scores, 0.95).item()
tau_mahalanobis = torch.quantile(
    mahalanobis_val_scores, 0.95
).item()

print(f"MSP threshold:         {tau_msp:.6f}")
print(f"MLS threshold:         {tau_mls:.6f}")
print(f"Energy threshold:      {tau_energy:.6f}")
print(f"Mahalanobis threshold: {tau_mahalanobis:.6f}")

MSP threshold:         0.449445
MLS threshold:         -4.392576
Energy threshold:      -4.861831
Mahalanobis threshold: 491.647705


In [12]:
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import pandas as pd


def evaluate_ood(known_features, unknown_features, score_fn, threshold):
    known_scores = np.array([
        score_fn(feature) for feature in known_features
    ])

    unknown_scores = np.array([
        score_fn(feature) for feature in unknown_features
    ])

    y_true = np.concatenate([
        np.zeros(len(known_scores)),
        np.ones(len(unknown_scores))
    ])

    scores = np.concatenate([
        known_scores,
        unknown_scores
    ])

    y_pred = (scores > threshold).astype(int)

    accuracy = accuracy_score(y_true, y_pred)
    auroc = roc_auc_score(y_true, scores)

    return accuracy, auroc


all_unknown_features = torch.cat([
    near_unknown_features,
    far_unknown_features
], dim=0)


methods = {
    "MSP": (
        msp_unknown_score,
        tau_msp
    ),
    "MLS": (
        mls_unknown_score,
        tau_mls
    ),
    "Energy": (
        energy_unknown_score,
        tau_energy
    ),
    "Mahalanobis": (
        mahalanobis_unknown_score,
        tau_mahalanobis
    )
}


comparisons = {
    "Known vs Near": near_unknown_features,
    "Known vs Far": far_unknown_features,
    "Known vs All Unknowns": all_unknown_features
}


results = []

for method_name, (score_fn, threshold) in methods.items():

    for comparison_name, unknown_features in comparisons.items():

        accuracy, auroc = evaluate_ood(
            test_features,
            unknown_features,
            score_fn,
            threshold
        )

        results.append({
            "Statistic": method_name,
            "Comparison": comparison_name,
            "Accuracy": accuracy,
            "AUROC": auroc
        })


results_df = pd.DataFrame(results)

results_df["Accuracy"] = (
    results_df["Accuracy"] * 100
).round(2)

results_df["AUROC"] = (
    results_df["AUROC"] * 100
).round(2)

display(results_df)

,Statistic,Comparison,Accuracy,AUROC
0,MSP,Known vs Near,84.10,74.55
1,MSP,Known vs Far,84.78,79.17
2,MSP,Known vs All Unknowns,76.24,76.86
3,MLS,Known vs Near,84.31,74.54
4,MLS,Known vs Far,85.55,82.72
5,MLS,Known vs All Unknowns,77.17,78.63
6,Energy,Known vs Near,84.43,74.45
7,Energy,Known vs Far,85.62,82.97
8,Energy,Known vs All Unknowns,77.35,78.71
9,Mahalanobis,Known vs Near,82.03,51.52


# GCSC – Strong Closed-Set Classifier + MLS

In [13]:
# Define the new transform
new_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# CIFAR-10 with new transform
cifar10_dataset_new = datasets.ImageFolder(
    root=cifar10_train_path,
    transform=new_transform,
)

train_dataset_new = Subset(
    cifar10_dataset_new,
    train_indices,
)

val_dataset_new = Subset(
    cifar10_dataset_new,
    val_indices,
)

test_dataset_new = Subset(
    cifar10_dataset_new,
    test_indices,
)


# CIFAR-100 with new transform
cifar100_dataset_new = CIFAR100Dataset(
    cifar100_data,
    cifar100_targets,
    transform=new_transform,
)

NearUnknownDataset_new = subset_by_classes(
    cifar100_dataset_new,
    near_unknown_classes,
    cifar100_class_to_idx,
)

FarUnknownDataset_new = subset_by_classes(
    cifar100_dataset_new,
    far_unknown_classes,
    cifar100_class_to_idx,
)


# DataLoaders
train_loader_new = DataLoader(
    train_dataset_new,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader_new = DataLoader(
    val_dataset_new,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader_new = DataLoader(
    test_dataset_new,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

NearUnknownDataLoader_new = DataLoader(
    NearUnknownDataset_new,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

FarUnknownDataLoader_new = DataLoader(
    FarUnknownDataset_new,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


print("CIFAR-10 Train:      ", len(train_dataset_new))
print("CIFAR-10 Validation: ", len(val_dataset_new))
print("CIFAR-10 Test:       ", len(test_dataset_new))
print("Near Unknown:        ", len(NearUnknownDataset_new))
print("Far Unknown:         ", len(FarUnknownDataset_new))

CIFAR-10 Train:       40500
CIFAR-10 Validation:  4500
CIFAR-10 Test:        5000
Near Unknown:         800
Far Unknown:          800


In [14]:
import torch
import torch.nn as nn
import torchvision.models as models

from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR

NUM_KNOWN_CLASSES = len(cifar10_idx_to_class)

resnet18_randaugment = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

resnet18_randaugment.fc = nn.Linear(
    resnet18_randaugment.fc.in_features,
    NUM_KNOWN_CLASSES
)

resnet18_randaugment = resnet18_randaugment.to(device)

loss_fn_randaugment = nn.CrossEntropyLoss()

optim_randaugment = SGD(
    resnet18_randaugment.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
)

scheduler_randaugment = CosineAnnealingLR(
    optim_randaugment,
    T_max=100,
    eta_min=1e-6,
)

In [15]:
num_epochs = 100

for epoch in range(num_epochs):

    resnet18_randaugment, train_loss = train_one_epoch(
        epoch,
        resnet18_randaugment,
        train_loader_new,
        optim_randaugment,
        loss_fn_randaugment
    )

    val_loss, val_accuracy = eval_model(
        resnet18_randaugment,
        val_loader_new,
        loss_fn_randaugment
    )

    scheduler_randaugment.step()

    current_lr = optim_randaugment.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}% | "
        f"LR: {current_lr:.6f}"
    )

torch.save(
    resnet18_randaugment.state_dict(),
    "resnet18_randaugment.pth"
)

print("RandAugment model saved successfully.")

Epoch [1/100] | Train Loss: 3.1835 | Val Loss: 2.4218 | Val Accuracy: 18.53% | LR: 0.099975
Epoch [2/100] | Train Loss: 2.1120 | Val Loss: 2.0430 | Val Accuracy: 22.69% | LR: 0.099901
Epoch [3/100] | Train Loss: 1.9139 | Val Loss: 1.9093 | Val Accuracy: 28.87% | LR: 0.099778
Epoch [4/100] | Train Loss: 1.7717 | Val Loss: 1.7716 | Val Accuracy: 36.07% | LR: 0.099606
Epoch [5/100] | Train Loss: 1.6582 | Val Loss: 1.6439 | Val Accuracy: 43.04% | LR: 0.099384
Epoch [6/100] | Train Loss: 1.5382 | Val Loss: 1.5539 | Val Accuracy: 43.40% | LR: 0.099114
Epoch [7/100] | Train Loss: 1.4498 | Val Loss: 1.5987 | Val Accuracy: 45.02% | LR: 0.098796
Epoch [8/100] | Train Loss: 1.3775 | Val Loss: 1.4320 | Val Accuracy: 48.56% | LR: 0.098429
Epoch [9/100] | Train Loss: 1.3201 | Val Loss: 1.5481 | Val Accuracy: 46.82% | LR: 0.098015
Epoch [10/100] | Train Loss: 1.2772 | Val Loss: 1.2861 | Val Accuracy: 54.13% | LR: 0.097553
Epoch [11/100] | Train Loss: 1.2429 | Val Loss: 1.3983 | Val Accuracy: 51.47% |

In [16]:
import copy
import torch
import torch.nn as nn

feature_extractor_randaugment = copy.deepcopy(
    resnet18_randaugment
)

feature_extractor_randaugment.fc = nn.Identity()

feature_extractor_randaugment = (
    feature_extractor_randaugment.to(device)
)

feature_extractor_randaugment.eval()


def extract_features_randaugment(
    model,
    loader,
    device
):
    features_list = []
    labels_list = []

    with torch.inference_mode():
        for images, labels in loader:

            images = images.to(device)

            features = model(images)

            features_list.append(
                features.cpu()
            )

            labels_list.append(
                labels.cpu()
            )

    features = torch.cat(
        features_list,
        dim=0
    )

    labels = torch.cat(
        labels_list,
        dim=0
    )

    return features, labels


train_features_randaugment, train_labels_randaugment = (
    extract_features_randaugment(
        feature_extractor_randaugment,
        train_loader_new,
        device
    )
)


val_features_randaugment, val_labels_randaugment = (
    extract_features_randaugment(
        feature_extractor_randaugment,
        val_loader_new,
        device
    )
)


test_features_randaugment, test_labels_randaugment = (
    extract_features_randaugment(
        feature_extractor_randaugment,
        test_loader_new,
        device
    )
)


near_unknown_features_randaugment, near_unknown_labels_randaugment = (
    extract_features_randaugment(
        feature_extractor_randaugment,
        NearUnknownDataLoader_new,
        device
    )
)


far_unknown_features_randaugment, far_unknown_labels_randaugment = (
    extract_features_randaugment(
        feature_extractor_randaugment,
        FarUnknownDataLoader_new,
        device
    )
)


print(
    "Train:",
    train_features_randaugment.shape
)

print(
    "Validation:",
    val_features_randaugment.shape
)

print(
    "Test:",
    test_features_randaugment.shape
)

print(
    "Near Unknown:",
    near_unknown_features_randaugment.shape
)

print(
    "Far Unknown:",
    far_unknown_features_randaugment.shape
)

Train: torch.Size([40500, 512])
Validation: torch.Size([4500, 512])
Test: torch.Size([5000, 512])
Near Unknown: torch.Size([800, 512])
Far Unknown: torch.Size([800, 512])


In [17]:
mls_val_scores_randaugment = torch.tensor([
    mls_unknown_score(feature)
    for feature in val_features_randaugment
])

tau_mls_randaugment = torch.quantile(
    mls_val_scores_randaugment,
    0.95
).item()

print(
    f"MLS threshold: "
    f"{tau_mls_randaugment:.6f}"
)

MLS threshold: -0.332963


In [18]:
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import pandas as pd
import torch


def mls_unknown_score_randaugment(feature):
    feature = feature.float().to(device)

    with torch.inference_mode():
        logits = resnet18_randaugment.fc(feature)

    score = -logits.max()

    return score.item()


def evaluate_ood(
    known_features,
    unknown_features,
    score_fn,
    threshold
):
    known_scores = np.array([
        score_fn(feature)
        for feature in known_features
    ])

    unknown_scores = np.array([
        score_fn(feature)
        for feature in unknown_features
    ])

    y_true = np.concatenate([
        np.zeros(len(known_scores)),
        np.ones(len(unknown_scores))
    ])

    scores = np.concatenate([
        known_scores,
        unknown_scores
    ])

    y_pred = (scores > threshold).astype(int)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    auroc = roc_auc_score(
        y_true,
        scores
    )

    return accuracy, auroc


all_unknown_features_randaugment = torch.cat([
    near_unknown_features_randaugment,
    far_unknown_features_randaugment
], dim=0)


comparisons = {
    "Known vs Near":
        near_unknown_features_randaugment,

    "Known vs Far":
        far_unknown_features_randaugment,

    "Known vs All Unknowns":
        all_unknown_features_randaugment
}


results = []

for comparison_name, unknown_features in comparisons.items():

    accuracy, auroc = evaluate_ood(
        test_features_randaugment,
        unknown_features,
        mls_unknown_score_randaugment,
        tau_mls_randaugment
    )

    results.append({
        "Statistic": "MLS",
        "Comparison": comparison_name,
        "Accuracy": accuracy,
        "AUROC": auroc
    })


results_df = pd.DataFrame(results)

results_df["Accuracy"] = (
    results_df["Accuracy"] * 100
).round(2)

results_df["AUROC"] = (
    results_df["AUROC"] * 100
).round(2)

display(results_df)

,Statistic,Comparison,Accuracy,AUROC
0,MLS,Known vs Near,86.21,70.74
1,MLS,Known vs Far,86.21,82.66
2,MLS,Known vs All Unknowns,75.76,76.70


# PROSER

In [19]:
import torch
import torch.nn as nn
from torchvision.models import resnet18


class PROSERResNet18(nn.Module):
    def __init__(self, baseline_path, num_known_classes=10, num_dummy_classes=5):
        super().__init__()

        self.num_known_classes = num_known_classes

        model = resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_known_classes)

        state = torch.load(baseline_path, map_location="cpu")
        state = state.get("model_state_dict", state.get("state_dict", state))
        model.load_state_dict(state)

        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.avgpool = model.avgpool

        self.fc = nn.Linear(
            model.fc.in_features,
            num_known_classes + num_dummy_classes
        )

        with torch.no_grad():
            self.fc.weight[:num_known_classes].copy_(model.fc.weight)
            self.fc.bias[:num_known_classes].copy_(model.fc.bias)

    def forward_to_layer2(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        return self.layer2(x)

    def forward_from_layer2(self, x):
        x = self.layer3(x)
        x = self.layer4(x)
        x = torch.flatten(self.avgpool(x), 1)
        return self.fc(x)

    def forward(self, x):
        return self.forward_from_layer2(
            self.forward_to_layer2(x)
        )

    def manifold_mixup(self, x1, x2, lam=0.1):
        h1 = self.forward_to_layer2(x1)
        h2 = self.forward_to_layer2(x2)

        if torch.is_tensor(lam) and lam.ndim == 1:
            lam = lam[:, None, None, None]

        h = lam * h1 + (1 - lam) * h2

        return self.forward_from_layer2(h)

    def known_logits(self, logits):
        return logits[:, :self.num_known_classes]

    def dummy_logits(self, logits):
        return logits[:, self.num_known_classes:]

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PROSERLoss(nn.Module):
    def __init__(
        self,
        num_known_classes=10,
        num_dummy_classes=5,
        beta=1.0,
    ):
        super().__init__()
        self.num_known_classes = num_known_classes
        self.num_dummy_classes = num_dummy_classes
        self.beta = beta

    def classifier_placeholder_loss(self, logits, targets):
        K = self.num_known_classes

        loss_known = F.cross_entropy(logits, targets)

        dummy_logits = logits[:, K:]
        dummy_idx = dummy_logits.detach().argmax(dim=1)
        dummy_targets = K + dummy_idx

        masked_logits = logits.clone()
        masked_logits[
            torch.arange(logits.size(0), device=logits.device),
            targets,
        ] = -torch.inf

        loss_dummy = F.cross_entropy(
            masked_logits,
            dummy_targets,
        )

        return loss_known + self.beta * loss_dummy

    def data_placeholder_loss(self, mixed_logits):
        K = self.num_known_classes

        dummy_logits = mixed_logits[:, K:]
        dummy_idx = dummy_logits.detach().argmax(dim=1)
        dummy_targets = K + dummy_idx

        return F.cross_entropy(
            mixed_logits,
            dummy_targets,
        )

    def forward(
        self,
        classifier_logits,
        classifier_targets,
        mixed_logits,
    ):
        loss_cls = self.classifier_placeholder_loss(
            classifier_logits,
            classifier_targets,
        )

        loss_data = self.data_placeholder_loss(
            mixed_logits,
        )

        loss = loss_cls + loss_data

        return {
            "loss": loss,
            "classifier_placeholder": loss_cls,
            "data_placeholder": loss_data,
        }

In [21]:
import random
import numpy as np
import torch

seed = 6304
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

model = PROSERResNet18(
    baseline_path="resnet18_baseline.pth"
).to(device)

loss_fn = PROSERLoss(
    num_known_classes=10,
    num_dummy_classes=5,
    beta=1.0
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=1e-3,
    momentum=0.9,
    weight_decay=5e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50
)


def train_proser_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        n = images.size(0) // 2

        x_cls = images[:n]
        y_cls = labels[:n]

        x_mix = images[n:]
        y_mix = labels[n:]

        logits_cls = model(x_cls)

        partners = torch.empty(
            len(y_mix),
            dtype=torch.long,
            device=device
        )

        for i in range(len(y_mix)):
            candidates = torch.where(y_mix != y_mix[i])[0]
            partners[i] = candidates[
                torch.randint(len(candidates), (1,), device=device)
            ]

        lam = torch.rand(len(x_mix), device=device)

        logits_mix = model.manifold_mixup(
            x_mix,
            x_mix[partners],
            lam
        )

        losses = loss_fn(
            logits_cls,
            y_cls,
            logits_mix
        )

        loss = losses["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


for epoch in range(50):
    train_loss = train_proser_epoch(
        model,
        train_loader,
        optimizer,
        loss_fn
    )

    scheduler.step()

    print(
        f"Epoch {epoch + 1:02d}/50 | "
        f"Loss: {train_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.6f}"
    )

/tmp/ipykernel_93719/4014832998.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(baseline_path, map_location="cpu")


Epoch 01/50 | Loss: 2.1272 | LR: 0.000999
Epoch 02/50 | Loss: 1.5472 | LR: 0.000996
Epoch 03/50 | Loss: 1.4638 | LR: 0.000991
Epoch 04/50 | Loss: 1.2872 | LR: 0.000984
Epoch 05/50 | Loss: 0.8242 | LR: 0.000976
Epoch 06/50 | Loss: 0.6678 | LR: 0.000965
Epoch 07/50 | Loss: 0.5829 | LR: 0.000952
Epoch 08/50 | Loss: 0.5534 | LR: 0.000938
Epoch 09/50 | Loss: 0.5316 | LR: 0.000922
Epoch 10/50 | Loss: 0.4971 | LR: 0.000905
Epoch 11/50 | Loss: 0.4846 | LR: 0.000885
Epoch 12/50 | Loss: 0.4808 | LR: 0.000864
Epoch 13/50 | Loss: 0.4796 | LR: 0.000842
Epoch 14/50 | Loss: 0.4594 | LR: 0.000819
Epoch 15/50 | Loss: 0.4545 | LR: 0.000794
Epoch 16/50 | Loss: 0.4498 | LR: 0.000768
Epoch 17/50 | Loss: 0.4410 | LR: 0.000741
Epoch 18/50 | Loss: 0.4411 | LR: 0.000713
Epoch 19/50 | Loss: 0.4198 | LR: 0.000684
Epoch 20/50 | Loss: 0.4210 | LR: 0.000655
Epoch 21/50 | Loss: 0.3899 | LR: 0.000624
Epoch 22/50 | Loss: 0.3954 | LR: 0.000594
Epoch 23/50 | Loss: 0.3804 | LR: 0.000563
Epoch 24/50 | Loss: 0.3744 | LR: 0

In [22]:
import copy
import torch
import torch.nn as nn

feature_extractor_proser = copy.deepcopy(model)

feature_extractor_proser.fc = nn.Identity()

feature_extractor_proser = (
    feature_extractor_proser.to(device)
)

feature_extractor_proser.eval()


def extract_features_proser(
    model,
    loader,
    device
):
    features_list = []
    labels_list = []

    with torch.inference_mode():
        for images, labels in loader:

            images = images.to(device)

            features = model(images)

            features_list.append(
                features.cpu()
            )

            labels_list.append(
                labels.cpu()
            )

    features = torch.cat(
        features_list,
        dim=0
    )

    labels = torch.cat(
        labels_list,
        dim=0
    )

    return features, labels

In [23]:
train_features_proser, train_labels_proser = (
    extract_features_proser(
        feature_extractor_proser,
        train_loader_new,
        device
    )
)

val_features_proser, val_labels_proser = (
    extract_features_proser(
        feature_extractor_proser,
        val_loader_new,
        device
    )
)

test_features_proser, test_labels_proser = (
    extract_features_proser(
        feature_extractor_proser,
        test_loader_new,
        device
    )
)

near_unknown_features_proser, near_unknown_labels_proser = (
    extract_features_proser(
        feature_extractor_proser,
        NearUnknownDataLoader_new,
        device
    )
)

far_unknown_features_proser, far_unknown_labels_proser = (
    extract_features_proser(
        feature_extractor_proser,
        FarUnknownDataLoader_new,
        device
    )
)

print("Train:", train_features_proser.shape)
print("Validation:", val_features_proser.shape)
print("Test:", test_features_proser.shape)
print("Near Unknown:", near_unknown_features_proser.shape)
print("Far Unknown:", far_unknown_features_proser.shape)

Train: torch.Size([40500, 512])
Validation: torch.Size([4500, 512])
Test: torch.Size([5000, 512])
Near Unknown: torch.Size([800, 512])
Far Unknown: torch.Size([800, 512])


In [24]:
model.eval()

def mls_unknown_score_proser(feature):
    with torch.inference_mode():

        feature = feature.to(device)

        logits = model.fc(feature)

        known_logits = logits[
            :model.num_known_classes
        ]

        score = -known_logits.max()

    return score.item()

In [25]:
mls_val_scores_proser = torch.tensor([
    mls_unknown_score_proser(feature)
    for feature in val_features_proser
])

tau_mls_proser = torch.quantile(
    mls_val_scores_proser,
    0.95
).item()

print(
    f"PROSER MLS threshold: "
    f"{tau_mls_proser:.6f}"
)

PROSER MLS threshold: -2.128862


In [26]:
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np
import pandas as pd
import torch


def evaluate_ood(
    known_features,
    unknown_features,
    score_fn,
    threshold
):
    known_scores = np.array([
        score_fn(feature)
        for feature in known_features
    ])

    unknown_scores = np.array([
        score_fn(feature)
        for feature in unknown_features
    ])

    y_true = np.concatenate([
        np.zeros(len(known_scores)),
        np.ones(len(unknown_scores))
    ])

    scores = np.concatenate([
        known_scores,
        unknown_scores
    ])

    y_pred = (
        scores > threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    auroc = roc_auc_score(
        y_true,
        scores
    )

    return accuracy, auroc


all_unknown_features_proser = torch.cat([
    near_unknown_features_proser,
    far_unknown_features_proser
], dim=0)


comparisons = {
    "Known vs Near":
        near_unknown_features_proser,

    "Known vs Far":
        far_unknown_features_proser,

    "Known vs All Unknowns":
        all_unknown_features_proser
}


results = []

for comparison_name, unknown_features in comparisons.items():

    accuracy, auroc = evaluate_ood(
        test_features_proser,
        unknown_features,
        mls_unknown_score_proser,
        tau_mls_proser
    )

    results.append({
        "Statistic": "MLS",
        "Comparison": comparison_name,
        "Accuracy": accuracy,
        "AUROC": auroc
    })


results_df = pd.DataFrame(results)

results_df["Accuracy"] = (
    results_df["Accuracy"] * 100
).round(2)

results_df["AUROC"] = (
    results_df["AUROC"] * 100
).round(2)

display(results_df)

,Statistic,Comparison,Accuracy,AUROC
0,MLS,Known vs Near,82.93,66.29
1,MLS,Known vs Far,82.69,65.30
2,MLS,Known vs All Unknowns,73.94,65.80


In [27]:
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import numpy as np
import pandas as pd
import torch


# ---------------------------------------------------------
# 1. Extract the actual PROSER OOD score
# ---------------------------------------------------------

def get_proser_outputs(loader):
    scores = []
    predictions = []

    model.eval()

    with torch.inference_mode():
        for images, _ in loader:
            images = images.to(device)

            # Your PROSER model directly returns 15 logits:
            # first 10 = known classes
            # last 5  = dummy classes
            logits = model(images)

            K = model.num_known_classes

            known_logits = logits[:, :K]
            dummy_logits = logits[:, K:]

            # Best known-class logit
            known_max = known_logits.max(dim=1).values

            # Best dummy-class logit
            dummy_max = dummy_logits.max(dim=1).values

            # PROSER OOD score
            # Higher => more unknown-like
            ood_score = dummy_max - known_max

            # Open-set prediction:
            # 0,...,K-1 = known classes
            # K = unknown
            open_logits = torch.cat(
                [known_logits, dummy_max.unsqueeze(1)],
                dim=1
            )

            pred = open_logits.argmax(dim=1)

            scores.append(ood_score.cpu())
            predictions.append(pred.cpu())

    return (
        torch.cat(scores).numpy(),
        torch.cat(predictions).numpy()
    )


# ---------------------------------------------------------
# 2. Generate scores explicitly
# ---------------------------------------------------------

val_scores_proser, val_preds_proser = get_proser_outputs(
    val_loader_new
)

known_scores_proser, known_preds_proser = get_proser_outputs(
    test_loader_new
)

near_scores_proser, near_preds_proser = get_proser_outputs(
    NearUnknownDataLoader_new
)

far_scores_proser, far_preds_proser = get_proser_outputs(
    FarUnknownDataLoader_new
)


# ---------------------------------------------------------
# 3. Calibrate threshold on KNOWN validation data
# ---------------------------------------------------------

tau_proser = np.quantile(
    val_scores_proser,
    0.95
)

print(f"PROSER threshold: {tau_proser:.6f}")

print(
    "Validation known rejection rate:",
    f"{(val_scores_proser > tau_proser).mean() * 100:.2f}%"
)

print(
    "Test known rejection rate:",
    f"{(known_scores_proser > tau_proser).mean() * 100:.2f}%"
)


# ---------------------------------------------------------
# 4. Evaluation
# ---------------------------------------------------------

def evaluate_proser(
    known_scores,
    unknown_scores,
    threshold
):
    # 0 = known
    # 1 = unknown

    y_true = np.concatenate([
        np.zeros(len(known_scores), dtype=int),
        np.ones(len(unknown_scores), dtype=int)
    ])

    scores = np.concatenate([
        known_scores,
        unknown_scores
    ])

    y_pred = (
        scores > threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    auroc = roc_auc_score(
        y_true,
        scores
    )

    known_rejection_rate = (
        known_scores > threshold
    ).mean()

    ood_detection_rate = (
        unknown_scores > threshold
    ).mean()

    return {
        "Accuracy": accuracy,
        "AUROC": auroc,
        "Known Rejection Rate": known_rejection_rate,
        "OOD Detection Rate": ood_detection_rate
    }


# ---------------------------------------------------------
# 5. Near, Far, and All Unknown
# ---------------------------------------------------------

all_unknown_scores_proser = np.concatenate([
    near_scores_proser,
    far_scores_proser
])

comparisons = {
    "Known vs Near": near_scores_proser,
    "Known vs Far": far_scores_proser,
    "Known vs All Unknowns": all_unknown_scores_proser
}


results = []

for comparison_name, unknown_scores in comparisons.items():

    metrics = evaluate_proser(
        known_scores_proser,
        unknown_scores,
        tau_proser
    )

    results.append({
        "Statistic": "PROSER",
        "Comparison": comparison_name,
        "Threshold": tau_proser,
        "Accuracy": metrics["Accuracy"] * 100,
        "AUROC": metrics["AUROC"] * 100,
        "Known Rejection (%)":
            metrics["Known Rejection Rate"] * 100,
        "OOD Detection (%)":
            metrics["OOD Detection Rate"] * 100
    })


results_df = pd.DataFrame(results)

numeric_columns = [
    "Threshold",
    "Accuracy",
    "AUROC",
    "Known Rejection (%)",
    "OOD Detection (%)"
]

results_df[numeric_columns] = (
    results_df[numeric_columns].round(2)
)

display(results_df)

PROSER threshold: 5.320008
Validation known rejection rate: 5.00%
Test known rejection rate: 5.94%


,Statistic,Comparison,Threshold,Accuracy,AUROC,Known Rejection (%),OOD Detection (%)
0,PROSER,Known vs Near,5.32,82.36,61.61,5.94,9.25
1,PROSER,Known vs Far,5.32,82.28,56.88,5.94,8.62
2,PROSER,Known vs All Unknowns,5.32,73.42,59.25,5.94,8.94


In [28]:
def evaluate_proser_csa(model, loader, device):
    """
    Closed-Set Accuracy (CSA) for PROSER.

    Uses only the 10 known-class logits.
    Dummy/placeholder logits are ignored.
    """

    model.eval()

    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # PROSER outputs:
            # [10 known-class logits | dummy logits]
            logits = model(images)

            # IMPORTANT:
            # Use ONLY the 10 known-class logits for CSA
            known_logits = logits[:, :10]

            # Predict among known classes only
            predictions = known_logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    csa = correct / total

    return csa


# Evaluate PROSER closed-set accuracy
proser_csa = evaluate_proser_csa(
    model,
    test_loader_new,
    device
)

print(f"PROSER Closed-Set Accuracy (CSA): {proser_csa * 100:.2f}%")

PROSER Closed-Set Accuracy (CSA): 72.30%
